# 06 — Evaluate and inspect mistakes
Reuse predictions for this checkpoint, or create them below. Run 05 first. Export every real validation box; score only labeled boxes.


In [1]:
# Cell purpose: Load the original best checkpoint and decide whether its evaluation predictions are reusable.
# Inputs: Saved samples, runs/best.pt and any previous evaluation metadata.
# Outputs: A verified checkpoint/sample pairing and prediction-cache reuse decision.

from pathlib import Path
import json
from config import CONFIG, ROOT, fingerprint, save_json, require_cache, run_stage, model_files

# Edit config.py once. Restart the kernel after changing it.
# Shared paths come from config.py; they are not hard-coded separately in each notebook.
CACHE = CONFIG['cache_dir']
RUN = CONFIG['run_dir']
CACHE.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)
print('Existing caches:', [p.name for p in CACHE.iterdir() if p.is_file()])

from IPython.display import display
import torch
import pandas as pd
from torch.utils.data import DataLoader
from model import BoxClassifier, collate, metrics, prediction_rows
# A missing prerequisite is created by its earlier notebook; an existing cache is reused.
require_cache(CACHE / 'samples.pt', '04_build_training_samples.ipynb')
assert (RUN / 'best.pt').exists(), 'Run notebook 05 first to train the classifier.'
data = torch.load(CACHE / 'samples.pt', map_location='cpu', weights_only=True)
checkpoint = torch.load(RUN / 'best.pt', map_location='cpu', weights_only=True)
# Reject a checkpoint whose sequence/features/split cache differs from the current samples.
assert checkpoint['samples_signature'] == data['signature'], 'Samples changed; rerun training'
PREDICTIONS = RUN / 'evaluation_predictions.json'
META = RUN / 'evaluation_meta.json'
import hashlib
# Tie reusable evaluation predictions to the exact saved checkpoint bytes.
checkpoint_hash = hashlib.sha256((RUN / 'best.pt').read_bytes()).hexdigest()
reuse = PREDICTIONS.exists() and META.exists() and json.loads(META.read_text())['checkpoint_sha256'] == checkpoint_hash
print('Reuse evaluated predictions:', reuse)

Existing caches: ['deberta.pt', 'discovery.json', 'magi.pt', 'samples.pt', 'sequences.json']
Reuse evaluated predictions: False


In [2]:
# Cell purpose: Evaluate or reuse predictions, score labeled boxes and export review tables.
# Inputs: The original validation split and best-checkpoint class probabilities.
# Outputs: Metrics JSON, all-real-box prediction CSV, labeled mistakes CSV and displayed summaries.

# Reuse predictions only after the checkpoint identity check has passed.
if reuse:
    rows = json.loads(PREDICTIONS.read_text(encoding='utf-8'))
else:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = BoxClassifier(checkpoint['visual_dim'], checkpoint['max_positions']).to(device)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    val = [data['samples'][i] for i in data['val_indices']]
    loader = DataLoader(val, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate)
    rows = []
    # Disable autograd for frozen feature extraction or evaluation to reduce memory use.
    with torch.inference_mode():
        for batch in loader:
            batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
            # Export real boxes only; prediction_rows excludes padding but retains ignored-label context boxes.
            rows.extend(prediction_rows(batch, model(batch).softmax(-1).cpu()))
    save_json(PREDICTIONS, rows)
    save_json(META, {'checkpoint_sha256': checkpoint_hash, 'epoch': checkpoint['epoch']})
report = metrics(rows)
# Exclude ignored real boxes from metrics/loss while retaining them in the full prediction export.
labeled = [r for r in rows if r['label'] != -100]
import math
weights = checkpoint['class_weights'].tolist()
report['val_loss'] = sum(-weights[r['label']] * math.log(max(
    r['p_spoken'] if r['label'] == 0 else r['p_narrated'], 1e-30)) for r in labeled) / len(labeled)
save_json(RUN / 'evaluation_metrics.json', report)
frame = pd.DataFrame(rows)
frame.to_csv(RUN / 'validation_predictions.csv', index=False)
# The mistake table includes only labeled boxes whose predicted class differs from GT.
errors = frame[(frame.label != -100) & (frame.label != frame.prediction)]
errors.to_csv(RUN / 'validation_errors.csv', index=False)
print(json.dumps(report, indent=2))
display(pd.crosstab(frame.loc[frame.label != -100, 'gt'], frame.loc[frame.label != -100, 'predicted_class']))
display(errors[['sequence_id', 'box_id', 'ocr', 'gt', 'predicted_class', 'p_spoken', 'p_narrated', 'tail_score']].head(30))

{
  "macro_f1": 0.8254247217340362,
  "balanced_accuracy": 0.8373819163292847,
  "SPOKEN": {
    "precision": 0.9859154929577465,
    "recall": 0.9824561403508771,
    "f1": 0.984182776801406,
    "support": 285,
    "p_true_mean": 0.9805102106910852,
    "p_true_median": 0.9999328851699829,
    "p_true_gt_0.9": 0.968421052631579,
    "p_true_lt_0.5": 0.017543859649122806
  },
  "NARRATED": {
    "precision": 0.6428571428571429,
    "recall": 0.6923076923076923,
    "f1": 0.6666666666666666,
    "support": 13,
    "p_true_mean": 0.621523513478818,
    "p_true_median": 0.909506618976593,
    "p_true_gt_0.9": 0.5384615384615384,
    "p_true_lt_0.5": 0.3076923076923077
  },
  "val_loss": 0.1379689028577222
}


predicted_class,NARRATED,SPOKEN
gt,,
NARRATED,9,4
SPOKEN,5,280


,sequence_id,box_id,ocr,gt,predicted_class,p_spoken,p_narrated,tail_score
12,seq_72fdfff4b4353a42,PG1:T13,Shirogane realizes the dilemma he is in!!,NARRATED,SPOKEN,0.833663,0.166337,4.069498e-04
14,seq_72fdfff4b4353a42,PG2:T2,"In other words, ""The One who Confesses Loses!!""",NARRATED,SPOKEN,0.998860,0.001140,0.000000e+00
296,seq_44b9a316b2919cfe,PG1:T7,"""Torso"" is estimated to be a dangerous A rate ...",NARRATED,SPOKEN,0.999583,0.000417,4.152107e-06
312,seq_44b9a316b2919cfe,PG3:T1,is taking my best efforts just to unify the team,SPOKEN,NARRATED,0.002787,0.997213,1.778734e-10
313,seq_44b9a316b2919cfe,PG3:T2,to say nothing of creating investigators like ...,SPOKEN,NARRATED,0.237680,0.762320,4.056475e-11
372,seq_ac5793f8168efdd1,PG2:T11,Rank 1 SASAKI.,SPOKEN,NARRATED,0.328566,0.671434,1.866098e-05
377,seq_ac5793f8168efdd1,PG2:T16,They are humans who had their natural enemy's ...,NARRATED,SPOKEN,0.956983,0.043016,4.779741e-04
387,seq_ac5793f8168efdd1,PG3:T9,My lover,SPOKEN,NARRATED,0.015438,0.984562,4.759317e-06
412,seq_41d68dfc426cc19e,PG2:T11,A single stroke of the sword with everything h...,SPOKEN,NARRATED,0.266330,0.733670,4.046637e-01


In [ ]:
# Cell purpose: Optional scratch cell; no operation is currently defined.
# Inputs: None.
# Outputs: None; this cell contains comments only.

